In [1]:
# ============================================================================
# 70-bit 有限域上的 Pohlig-Hellman 攻击（手动实现）
# ============================================================================
import time
from sage.groups.generic import discrete_log
from sage.arith.misc import CRT

def pohlig_hellman(P, Q, order, factors):
    """
    手动实现 Pohlig-Hellman 算法。
    P: 生成元 (已知阶为 order)
    Q: 待求对数 Q = kP
    order: P 的阶（整数）
    factors: order 的素因子分解列表 [(p1, e1), (p2, e2), ...]
    返回 k mod order
    """
    k_crt = []
    moduli = []
    for pi, ei in factors:
        # 处理每个素因子幂次
        # 计算当前子群阶 pi^ei
        pe = pi**ei
        # 计算辅助点：将问题降到 pi^ei 子群
        # 使用 P_i = (order/pe)*P，阶为 pe
        Pi = (order // pe) * P
        Qi = (order // pe) * Q
        # 求解离散对数在阶为 pe 的子群中
        # 可以使用 Pohlig-Hellman 递归或直接用小步大步，这里因为 pe 可能很小，直接用 discrete_log
        # 注意：这里的 discrete_log 会自动使用适合小阶的算法，不会出错
        ki = discrete_log(Qi, Pi, operation='+')
        k_crt.append(ki)
        moduli.append(pe)
    # 中国剩余定理组合
    k = CRT(k_crt, moduli)
    return k

# 1. 生成 70-bit 素数 p
p = random_prime(2^99, lbound=2^98)
print(f"素数 p = {p} (比特长度: {p.nbits()})")
F = GF(p)

# 2. 寻找一条阶光滑的椭圆曲线（所有素因子比特长度 ≤ 20）
print("正在搜索阶光滑的椭圆曲线（最大素因子 ≤ 2^20）...")
found = False
max_attempts = 5000
for attempt in range(max_attempts):
    a = F.random_element()
    b = F.random_element()
    try:
        E = EllipticCurve(F, [a, b])
        order = E.order()
        if order == p:
            continue
        factors = order.factor()
        max_prime_bits = max([f[0].nbits() for f in factors])
        if max_prime_bits <= 20:
            found = True
            break
    except:
        continue

if not found:
    print(f"经过 {max_attempts} 次尝试未找到阶光滑曲线。使用小素数 p=101 上的预置示例。")
    p_small = 101
    F_small = GF(p_small)
    E = EllipticCurve(F_small, [2, 3])
    order = E.order()
    factors = order.factor()
    print(f"曲线: y^2 = x^3 + 2x + 3 over F_{p_small}")
    print(f"阶: {order} = {factors}")
    P = E.gens()[0]
    k_true = 12345 % order
    Q = k_true * P
    start = time.time()
    k_found = pohlig_hellman(P, Q, order, list(factors))
    elapsed = time.time() - start
    print(f"预期私钥: {k_true}")
    print(f"求解私钥: {k_found}")
    print(f"Pohlig-Hellman 破解时间: {elapsed:.6f} 秒")
else:
    print(f"找到光滑曲线，尝试次数: {attempt+1}")
    print(f"曲线: y^2 = x^3 + {a}x + {b}")
    print(f"阶: {order} = {factors}")
    P = E.gens()[0]
    # 确保生成元的阶就是整个阶（通常是的）
    print(f"生成元阶: {P.order()}")
    k_true = randint(1, order-1)
    Q = k_true * P
    start = time.time()
    k_found = pohlig_hellman(P, Q, order, list(factors))
    elapsed = time.time() - start
    print(f"预期私钥: {k_true}")
    print(f"求解私钥: {k_found}")
    print(f"Pohlig-Hellman 破解时间: {elapsed:.4f} 秒")

素数 p = 532124079109448751212146710899 (比特长度: 99)
正在搜索阶光滑的椭圆曲线（最大素因子 ≤ 2^20）...
找到光滑曲线，尝试次数: 3987
曲线: y^2 = x^3 + 63910364058510815946995869339x + 216102714601431725567927550578
阶: 532124079109448838793570826480 = 2^4 * 5 * 19 * 127 * 563 * 1733 * 9173 * 15493 * 34031 * 584167
生成元阶: 266062039554724419396785413240
预期私钥: 99112727307597138553566911203
求解私钥: 99112727307597138553566911203
Pohlig-Hellman 破解时间: 0.0470 秒
